# 00 — Project overview

**Objectives**
1. Understand what the Analytics Agent is, how it is structured, and which stage is delivered.
2. Verify that the local environment (virtual environment, kernel, `src/` package) is wired correctly.

**Prerequisites:** completed the setup in `README.md` §2 (virtual environment created, `pip install -r requirements.txt` and `pip install -e .`, kernel selected in VS Code).

**Configuration needed:** none for this notebook. Google Cloud, Gemini and Langfuse are configured in notebooks 03, 05 and 07.

## Architecture in one paragraph

Business questions flow through intent resolution → schema/metric retrieval → SQL generation → a deterministic SQL validator → an LLM SQL reviewer → a BigQuery dry run with byte caps → read-only execution → result validation → an LLM answer verifier → the final answer with SQL and caveats. A separate deterministic data-quality engine checks the warehouse and an LLM reviewer explains its findings. Full diagrams: `docs/01_requirements_architecture.md`.

| Stage | Notebook(s) | Status |
|---|---|---|
| 1 Design | 00, 01 | delivered |
| 2 Synthetic data | 02 | delivered |
| 3 BigQuery upload + connection | 03 | next |
| 4 Gemini setup | (05 setup cells) | planned |
| 5 Data-quality engine | 04 | planned |
| 6 Reviewers + analytics agent | 05, 06 | planned |
| 7 Orchestration + Langfuse | 07 | planned |
| 8 Evaluation + demo | 08 | planned |

## Step 1 — Locate the project root and make `src/` importable

Why: notebooks live in `notebooks/` but the package lives in `src/`. This cell makes imports work even if VS Code starts the kernel in a different folder.

In [1]:
# Locate the project root (folder containing pyproject.toml) so the notebook works
# no matter which folder VS Code starts the kernel in, then make `src/` importable.
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    if root.parent == root:
        raise RuntimeError("Could not find pyproject.toml - open the analytics-agent folder in VS Code.")
    root = root.parent
os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
print("Project root:", root)
print("Python:", sys.version.split()[0], "| executable:", sys.executable)

Project root: /home/claude/analytics-agent
Python: 3.12.3 | executable: /home/claude/.venv/bin/python


## Step 2 — Check the interpreter is the project virtual environment

**Checkpoint:** the executable path should contain `.venv`. If not, pick the right kernel (top-right of the notebook → *Select Kernel* → the `.venv` interpreter).

In [2]:
in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
print("Running inside a virtual environment:", in_venv)
if not in_venv:
    print("WARNING: select the project's .venv kernel (README section 2).")

Running inside a virtual environment: True


## Step 3 — Check required packages and versions

**Expected output:** each package prints the version pinned in `requirements.txt`.

In [3]:
import importlib
expected = {"numpy": "2.2.6", "pandas": "2.2.3", "pyarrow": "19.0.1", "yaml": "6.0.2"}
problems = []
for mod, want in expected.items():
    try:
        have = importlib.import_module(mod).__version__
    except ImportError:
        have = "NOT INSTALLED"
    flag = "ok" if have == want else "MISMATCH"
    if flag != "ok":
        problems.append(mod)
    print(f"{mod:<10} expected {want:<8} found {have:<14} {flag}")
assert not problems, f"Fix with: pip install -r requirements.txt  (problems: {problems})"

numpy      expected 2.2.6    found 2.2.6          ok


pandas     expected 2.2.3    found 2.2.3          ok
pyarrow    expected 19.0.1   found 19.0.1         ok
yaml       expected 6.0.2    found 6.0.2          ok


## Step 4 — Import the project package

**Checkpoint:** prints the seven approved tables.

In [4]:
from analytics_agent import schema_registry as sr
print("Approved tables:", list(sr.TABLE_LOAD_ORDER))
assert len(sr.TABLES) == 7

Approved tables: ['customers', 'products', 'orders', 'order_items', 'payments', 'shipments', 'refunds']


## Troubleshooting

| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: analytics_agent` | Run Step 1, or `pip install -e .` from the project root with the venv active. |
| Wrong versions | The kernel is not the project venv. Select it, or reinstall: `pip install -r requirements.txt`. |
| `Could not find pyproject.toml` | Open the `analytics-agent` folder itself in VS Code (*File → Open Folder*). |

## Summary and next steps
The environment is ready. Next: **01_dataset_design** to review the data model, then **02_generate_synthetic_data**.

## References
- VS Code Jupyter: https://code.visualstudio.com/docs/datascience/jupyter-notebooks
- Python venv: https://docs.python.org/3/library/venv.html